# UVA + CoT/LLM wrapper → LIBERO-10 평가 (Colab)

동결된 UVA(`libero10.ckpt`) 위에 **추론 시점 CoT 오케스트레이션**을 씌워 LIBERO-10을 평가합니다.

| 모드 | 설명 |
|------|------|
| `no_cot=True` | 논문 baseline (`eval_sim.py`와 동일) |
| `planner="rule"` | 규칙 기반 CoT → `language_goal` 재작성 (API 불필요) |
| `planner="llm"` | OpenAI vision CoT (`gpt-4o-mini` 등, **API 키 필요**) |

**비교 목표**: baseline vs rule-CoT vs LLM-CoT 의 `test_mean_score`.

> 이미 `colab_libero10_eval.ipynb`로 4b까지 설치했다면 **셀 0 → 8(OpenAI 키) → 9(함수) → 10~** 만 실행해도 됩니다.

> **런타임**: GPU (L4/T4/A100). `MUJOCO_GL=egl` 필수. LLM 모드는 Colab Secrets에 `OPENAI_API_KEY` 등록.

## 0. Python 3.10 설치

★ 처음 한 번만 실행. 자동 재시작 후 Python 버전 확인하고 다음 셀부터 진행.

In [ ]:
# 0. Python 3.10 설치 (condacolab)
# ★ 최대 2회 자동 재시작 — 마지막에 "Python 3.10 OK" 뜨면 다음 셀로
import sys, shutil
print('Python:', sys.version)

if sys.version_info >= (3, 11):
    if shutil.which('conda'):
        # condacolab 재시작 후 → conda로 Python 3.10 강제 설치
        print('conda 감지 → Python 3.10 설치 중...')
        import subprocess
        subprocess.run(['conda', 'install', '-y', '-q', 'python=3.10'], check=True)
        print('완료 → 재시작')
        import os; os.kill(os.getpid(), 9)
    else:
        # 최초 실행 → condacolab 설치 후 자동 재시작
        !pip install -q condacolab
        import condacolab
        condacolab.install()
else:
    print('Python 3.10 OK → 아래 셀 계속 실행')

## 1. 런타임 체크 + headless GL

In [ ]:
import os, subprocess
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["MKL_THREADING_LAYER"] = "GNU"
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 2. Google Drive (선택)

In [ ]:
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT = "/content/drive/MyDrive/uva_libero"
    os.makedirs(PERSIST_ROOT, exist_ok=True)
    print("Drive:", PERSIST_ROOT)
else:
    PERSIST_ROOT = None
    print("Drive 미사용")

## 3. apt 패키지 (EGL / MuJoCo)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libegl1 libegl1-mesa libgles2-mesa libgl1-mesa-glx \
    libosmesa6 libosmesa6-dev libglfw3 libglew-dev patchelf ffmpeg > /dev/null
# mujoco_py 빌드 도구
!apt-get install -y -qq python3-dev build-essential pkg-config \
    libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libswresample-dev > /dev/null
# MuJoCo 2.1 (mujoco_py 런타임용)
import pathlib
_mj_dir = pathlib.Path.home() / '.mujoco' / 'mujoco210'
if not _mj_dir.exists():
    !wget -q https://github.com/deepmind/mujoco/releases/download/2.1.0/mujoco210-linux-x86_64.tar.gz -O /tmp/mujoco210.tar.gz
    !tar -xzf /tmp/mujoco210.tar.gz -C ~/.mujoco
    print('MuJoCo 2.1 설치 완료')
else:
    print('MuJoCo 2.1 이미 설치됨')

## 4. 레포 클론

In [ ]:
#import os

UVA_REPO_URL    = "https://github.com/zser99/Unified-Video-Action-with-LLM.git"
UVA_REPO_BRANCH = "main"

%cd /content
if not os.path.isdir("/content/unified_video_action"):
    !git clone --depth 1 -b {UVA_REPO_BRANCH} {UVA_REPO_URL} unified_video_action
if not os.path.isdir("/content/LIBERO"):
    !git clone --depth 1 https://github.com/Lifelong-Robot-Learning/LIBERO.git
!ls /content/unified_video_action/unified_video_action/cot 2>/dev/null || echo "WARNING: cot/ folder missing — fork URL 확인"

## 5. 패키지 설치

In [ ]:
import sys, subprocess, os
assert sys.version_info[:2] == (3, 10), f"Python 3.10 필요 (현재: {sys.version[:6]}) — 셀 0 재실행"

def _pip(*args, check=True):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    print("$", " ".join(cmd))
    return subprocess.run(cmd, capture_output=True, text=True, check=check)

print("Python:", sys.version)
_pip("install", "-q", "gym==0.21.0", check=False)
import gym; print("gym OK:", gym.__version__)

_pip("install", "-q",
     "numpy==1.26.4", "scipy==1.11.4", "numba==0.60.0",
     "hydra-core==1.2.0", "omegaconf==2.2.3", "dill==0.3.7", "einops==0.6.1",
     "diffusers==0.18.2", "transformers==4.28.0", "huggingface_hub==0.25.2",
     "timm==0.9.7", "accelerate==1.0.1", "zarr==2.16.1", "numcodecs==0.11.0",
     "imagecodecs==2024.12.30", "kornia==0.8.0", "opencv-python==4.11.0.86",
     "h5py", "wandb==0.18.3", "gdown==5.2.0", "click", "pandas", "openai>=1.40.0")
_pip("install", "-q", "av", check=False)
_pip("install", "-q", "mujoco==3.1.6", "robosuite==1.4.1")
_pip("install", "-q", "robomimic==0.3.0", check=False)
# mujoco_py 2.1.x (robomimic env_robosuite.py 용)
os.environ["MUJOCO_PY_MUJOCO_PATH"] = os.path.expanduser("~/.mujoco/mujoco210")
os.environ["LD_LIBRARY_PATH"] = (os.environ.get("LD_LIBRARY_PATH", "") +
                                  ":" + os.path.expanduser("~/.mujoco/mujoco210/bin"))
_pip("install", "-q", "mujoco-py==2.1.2.14")
_pip("install", "-q", "-e", "/content/LIBERO", "--no-deps")
_pip("install", "-q", "-e", "/content/unified_video_action", "--no-deps")
print("5. 패키지 설치 완료")

## 6. 체크포인트 (~3 GB)

In [ ]:
import os
os.environ.setdefault("MUJOCO_GL", "egl")
%cd /content/unified_video_action
os.makedirs("checkpoints", exist_ok=True)
CKPT = "checkpoints/libero10.ckpt"
if not os.path.isfile(CKPT):
    !gdown 11c2VrmaRp48yw__5A5xpcu8EPzkexHSi -O {CKPT}
!ls -lh checkpoints

## 7. LIBERO-10 hdf5

In [ ]:
import glob
%cd /content/unified_video_action
os.makedirs("data/libero_10", exist_ok=True)
if len(glob.glob("data/libero_10/*.hdf5")) < 10:
    !gdown 1_6Kc7e-s30MblbX8YjpxSofe9ZRPk3xv -O data/libero_10_raw.zip
    !unzip -oq data/libero_10_raw.zip -d data/
    for cand in ["libero10", "Libero10"]:
        p = f"data/{cand}"
        if os.path.isdir(p):
            !mv {p}/*.hdf5 data/libero_10/ 2>/dev/null || true
    !rm -f data/libero_10_raw.zip
hdf5 = sorted(glob.glob("data/libero_10/*.hdf5"))
print(len(hdf5), "files"); assert len(hdf5) == 10

## 8. 초기화 ← 재시작할 때마다 이 셀 하나만 실행

In [ ]:
"""재시작 후 초기화 — 이 셀 하나로 환경·shim 전부 처리"""
import os, sys, subprocess, types, urllib.request, importlib
from pathlib import Path

# ── 1. 환경변수 ──────────────────────────────────────────────
os.environ["MUJOCO_GL"]          = "egl"
os.environ["PYOPENGL_PLATFORM"]  = "egl"
os.environ["MKL_THREADING_LAYER"] = "GNU"
os.chdir("/content/unified_video_action")

# ── 2. sys.path ───────────────────────────────────────────────
for p in ("/content/LIBERO", "/content/unified_video_action"):
    if p not in sys.path:
        sys.path.insert(0, p)

# ── 3. 필수 패키지 (pip 설치가 안 돼 있을 때만 설치) ───────────
for pkg, pip_name in [("bddl", "bddl==1.0.1"), ("easydict", "easydict")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=False)

# ── 4. mujoco_py 환경변수 + 임포트 ──────────────────────────
import pathlib as _pl
_mj210 = _pl.Path.home() / '.mujoco' / 'mujoco210'
if _mj210.exists():
    os.environ.setdefault('MUJOCO_PY_MUJOCO_PATH', str(_mj210))
    _lib = str(_mj210 / 'bin')
    if _lib not in os.environ.get('LD_LIBRARY_PATH', ''):
        os.environ['LD_LIBRARY_PATH'] = os.environ.get('LD_LIBRARY_PATH', '') + ':' + _lib
try:
    import mujoco_py
    print(f'mujoco_py: {mujoco_py.__version__} OK')
except ImportError as e:
    print(f'mujoco_py 로드 실패: {e}')
    raise

# ── 5. pytorch3d shim ─────────────────────────────────────────
import torch, numpy as np
from scipy.spatial.transform import Rotation as R

def _to_numpy(x):
    if isinstance(x, torch.Tensor): return x.detach().cpu().numpy(), x
    return np.asarray(x), None

def _to_like(arr, like):
    if like is None: return arr
    return torch.from_numpy(arr).to(device=like.device, dtype=like.dtype)

def axis_angle_to_matrix(x):
    xn, like = _to_numpy(x)
    return _to_like(R.from_rotvec(xn.reshape(-1,3)).as_matrix().reshape(*xn.shape[:-1],3,3), like)

def matrix_to_axis_angle(x):
    xn, like = _to_numpy(x)
    return _to_like(R.from_matrix(xn.reshape(-1,3,3)).as_rotvec().reshape(*xn.shape[:-2],3), like)

def matrix_to_rotation_6d(x):
    if isinstance(x, torch.Tensor): return x[...,:3,:2].reshape(*x.shape[:-2],6)
    xn,_ = _to_numpy(x); return xn[...,:3,:2].reshape(*xn.shape[:-2],6)

def rotation_6d_to_matrix(x):
    if isinstance(x, torch.Tensor):
        a1,a2 = x[...,0:3], x[...,3:6]
        b1 = torch.nn.functional.normalize(a1, dim=-1)
        b2 = torch.nn.functional.normalize(a2-(b1*a2).sum(-1,keepdim=True)*b1, dim=-1)
        return torch.stack((b1, b2, torch.cross(b1,b2,dim=-1)), dim=-1)
    xn = np.asarray(x); a1,a2 = xn[...,0:3], xn[...,3:6]
    b1 = a1/(np.linalg.norm(a1,axis=-1,keepdims=True)+1e-8)
    b2 = a2-(b1*a2).sum(axis=-1,keepdims=True)*b1
    b2 = b2/(np.linalg.norm(b2,axis=-1,keepdims=True)+1e-8)
    return np.stack((b1, b2, np.cross(b1,b2,axis=-1)), axis=-1)

_pt = types.ModuleType("pytorch3d.transforms")
for _fn in [axis_angle_to_matrix, matrix_to_axis_angle, matrix_to_rotation_6d, rotation_6d_to_matrix]:
    setattr(_pt, _fn.__name__, _fn)
_p3d = types.ModuleType("pytorch3d"); _p3d.transforms = _pt
sys.modules.update({"pytorch3d": _p3d, "pytorch3d.transforms": _pt})
print("pytorch3d: shim OK")

# ── 6. LiberoImageRunner 최신 파일 확보 ───────────────────────
RUNNER_DIR = Path("/content/unified_video_action/unified_video_action/env_runner")
UPSTREAM = "https://raw.githubusercontent.com/ShuangLI59/unified_video_action/main/unified_video_action/env_runner"
for _name in ("libero_image_runner.py", "libero_bddl_mapping.py"):
    urllib.request.urlretrieve(f"{UPSTREAM}/{_name}", RUNNER_DIR / _name)

import unified_video_action.env_runner.libero_image_runner as _lir
importlib.reload(_lir)
from unified_video_action.env_runner.libero_image_runner import LiberoImageRunner
print("LiberoImageRunner: OK")

# ── 7. CoT 모듈 확인 ──────────────────────────────────────────
from unified_video_action.cot.factory import create_planner
from unified_video_action.policy.cot_orchestrated_policy import CoTOrchestratedPolicy
print("CoT modules: OK")
print("━" * 45)
print("초기화 완료 → 셀 8 (OpenAI 키) → 셀 9 (함수 정의) → 셀 10 (실험)")

## 9. OpenAI API 키 (LLM CoT용)

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("OPENAI_API_KEY: loaded from Colab Secrets (", len(os.environ["OPENAI_API_KEY"]), "chars)")
except Exception as e:
    print("Secrets 없음:", e)
    print("수동 설정: os.environ['OPENAI_API_KEY'] = 'sk-...'")
# os.environ["OPENAI_API_KEY"] = "sk-..."  # 필요 시 주석 해제

## 10. 평가 함수 정의

In [ ]:
import os, sys, json, glob, random, pathlib, dill, hydra, torch, numpy as np, wandb
from omegaconf import open_dict

%cd /content/unified_video_action
sys.path.insert(0, "/content/unified_video_action")

from unified_video_action.workspace.base_workspace import BaseWorkspace
from unified_video_action.env_runner.base_image_runner import BaseImageRunner
from unified_video_action.cot.factory import create_planner
from unified_video_action.policy.cot_orchestrated_policy import CoTOrchestratedPolicy


def _instantiate_libero_runners(cfg, output_dir, task_subset):
    hdf5_files = sorted(glob.glob(cfg.task.dataset.dataset_path + "/*hdf5"))
    if task_subset:
        hdf5_files = [f for f in hdf5_files if any(s in os.path.basename(f) for s in task_subset)]
        assert hdf5_files, f"task_subset {task_subset} matched nothing"
    print(f"Tasks ({len(hdf5_files)}):")
    for f in hdf5_files:
        print(" ", os.path.basename(f))
    runners = []
    for f in hdf5_files:
        r = hydra.utils.instantiate(cfg.task.env_runner, task_dir=f, output_dir=output_dir)
        assert isinstance(r, BaseImageRunner)
        runners.append(r)
    return runners


def run_libero10_cot_eval(
    checkpoint="checkpoints/libero10.ckpt",
    output_dir="outputs/libero10_cot",
    device="cuda:0",
    n_test=1,
    n_train=0,
    n_envs=1,
    n_test_vis=0,
    max_steps=None,
    task_subset=None,
  # --- CoT ---
    no_cot=False,
    planner="llm",
    llm_model="gpt-4o-mini",
    replan_every=8,
    num_candidates=1,
    candidate_strategy="first",
    verbose_cot=False,
    no_llm_fallback=False,
):
    pathlib.Path(output_dir).mkdir(parents=True, exist_ok=True)
    if device.startswith("cuda") and not torch.cuda.is_available():
        print("WARNING: no CUDA, using cpu"); device = "cpu"

    payload = torch.load(open(checkpoint, "rb"), pickle_module=dill, map_location="cpu")
    cfg = payload["cfg"]
    seed = cfg.training.seed
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)

    with open_dict(cfg):
        cfg.output_dir = output_dir
        cfg.task.env_runner.n_test = int(n_test)
        cfg.task.env_runner.n_train = int(n_train)
        cfg.task.env_runner.n_envs = int(n_envs)
        cfg.task.env_runner.n_test_vis = min(n_test_vis, n_test)
        cfg.task.env_runner.n_train_vis = 0
        if max_steps is not None:
            cfg.task.env_runner.max_steps = int(max_steps)

    workspace = hydra.utils.get_class(cfg.model._target_)(cfg, output_dir=output_dir)
    workspace.load_payload(payload, exclude_keys=None, include_keys=None, strict=False)
    inner = workspace.ema_model
    inner.to(device); inner.eval()

    policy = inner
    cot_meta = {"cot_enabled": False}
    if not no_cot:
        cot_planner = create_planner(
            planner, model=llm_model, fallback_rule_based=not no_llm_fallback
        )
        policy = CoTOrchestratedPolicy(
            inner_policy=inner,
            planner=cot_planner,
            replan_every=replan_every,
            num_candidates=num_candidates,
            candidate_strategy=candidate_strategy,
            verbose=verbose_cot,
        )
        cot_meta = {
            "cot_enabled": True,
            "planner": planner,
            "replan_every": replan_every,
            "num_candidates": num_candidates,
            "candidate_strategy": candidate_strategy,
        }
        if planner == "llm":
            cot_meta["llm_model"] = llm_model

    env_runners = _instantiate_libero_runners(cfg, output_dir, task_subset)
    step_log = {}
    for er in env_runners:
        step_log.update(er.run(policy))
        try: er.env.close()
        except Exception: pass

    test_scores = {k: v for k, v in step_log.items() if "test/" in k and "_mean_score" in k}
    step_log["test_mean_score"] = float(np.mean(list(test_scores.values()))) if test_scores else float("nan")

    json_log = {}
    for k, v in step_log.items():
        if isinstance(v, wandb.sdk.data_types.video.Video):
            json_log[k] = v._path
        else:
            try: json.dumps(v); json_log[k] = v
            except TypeError: json_log[k] = str(v)
    json_log.update(cot_meta)
    json_log["device"] = device
    tag = "baseline" if no_cot else planner
    out_path = os.path.join(output_dir, f"eval_cot_{tag}_{os.path.basename(checkpoint)}.json")
    with open(out_path, "w") as f:
        json.dump(json_log, f, indent=2, sort_keys=True)
    print("Saved", out_path)
    print("test_mean_score:", step_log["test_mean_score"])
    return step_log, json_log

# lr_scheduler.py 패치 — 구버전 import 스타일일 때만 (repo가 이미 패치됐으면 no-op)
from pathlib import Path
lr = Path("/content/unified_video_action/unified_video_action/model/common/lr_scheduler.py")
if lr.exists():
    text = lr.read_text()
    if "from diffusers.optimization import (\n    Union," in text:
        lr.write_text(text.replace(
            "from diffusers.optimization import (\n    Union,\n    SchedulerType,\n    Optional,\n    Optimizer,\n    TYPE_TO_SCHEDULER_FUNCTION,\n)",
            "from typing import Optional, Union\n\nfrom diffusers.optimization import (\n    SchedulerType,\n    Optimizer,\n    TYPE_TO_SCHEDULER_FUNCTION,\n)",
        ))
        print("patched lr_scheduler.py (old style)")
    else:
        print("lr_scheduler.py already patched")

## 11. Smoke — 1 task × 1 ep (baseline / rule / LLM 비교)

In [ ]:
SMOKE_TASK = ["moka_pot"]
SMOKE_KW = dict(n_test=1, n_train=0, n_envs=1, n_test_vis=0, max_steps=300, task_subset=SMOKE_TASK)

step_baseline, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/baseline",
    no_cot=True,
    **SMOKE_KW,
)

step_rule, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/rule",
    planner="rule",
    verbose_cot=True,
    **SMOKE_KW,
)

step_llm, _ = run_libero10_cot_eval(
    output_dir="outputs/cot_smoke/llm",
    planner="llm",
    llm_model="gpt-4o-mini",
    verbose_cot=True,
    **SMOKE_KW,
)

## 12. Light — 10 tasks × 3 ep (LLM CoT, 논문 30-test 규모)

In [ ]:
step_light_llm, json_light_llm = run_libero10_cot_eval(
    output_dir="outputs/cot_light/llm",
    planner="llm",
    llm_model="gpt-4o-mini",
    n_test=3,
    n_envs=1,
    n_test_vis=0,
    verbose_cot=False,
)
print("LLM light mean:", step_light_llm.get("test_mean_score"))

## 13. 결과 비교표

In [ ]:
import pandas as pd

rows = []
for name, log in [
    ("baseline", locals().get("step_baseline")),
    ("rule-CoT", locals().get("step_rule")),
    ("LLM-CoT smoke", locals().get("step_llm")),
    ("LLM-CoT light", locals().get("step_light_llm")),
]:
    if log is None:
        continue
    rows.append({"mode": name, "test_mean_score": log.get("test_mean_score", float("nan"))})

if rows:
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print("\n논문 Table I (UVA baseline, 50 ep): 0.90")
    print("논문 Suppl. VIII (30 ep): 0.93")
else:
    print("아직 평가 결과 없음 — 셀 10 이상 실행")

## 트러블슈팅
- **`cot/` 없음** → 셀 3 fork URL이 LLM wrapper 브랜치인지 확인.
- **LLM이 rule로 fallback** → `OPENAI_API_KEY` 미설정. 셀 8 + trace에 `[LLM skipped]` 확인.
- **gym 설치 실패** → `colab_libero10_eval.ipynb`와 동일: 4a → 재시작 → 4b.
- **API 비용** → smoke는 1 task·1 ep; light는 replan마다 1회 API 호출 × (steps/8) × 30 ep.